# INF6083 – Projet P3
## Tâche 0 : Évaluation du filtrage basé sur le contenu (P2)  
## Tâche 1 : Filtrage collaboratif item-based

**Équipe 7**

---

**Sous-ensemble D** : `temporal_pre_split` (Amazon Reviews 2023 – Books)  
→ 22 007 items · 6 739 utilisateurs · split temporel par utilisateur

**Artefacts P2 réutilisés depuis le projet 2** (`/sysdereco_devoir2/data/joining/temporal_pre_split/`) :
- `train_interactions.parquet` / `test_interactions.parquet` — splits train/test
- `books_representation_sparse.npz` — matrice TF-IDF items (items × features)
- `user_profiles_tfidf.npz` — profils utilisateurs (moyenne pondérée des vecteurs items)
- `user_ids.npy`, `item_ids.npy`, `item_titles.npy` — tables d'identifiants
- `top_n_indices_10.npy` — Top-10 content-based déjà calculé par `scripts/task1/similarity.py`

**Métriques communes** : Precision@10, Recall@10, NDCG@10 (gradué + binaire), MAP@10, MRR@10, Hit Rate@10, ILD, Couverture catalogue

---
## 0. Configuration et imports

In [ ]:
import gc
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import load_npz, csr_matrix
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# ── Chemins ────────────────────────────────────────────────────────────────
# Ce notebook est dans Devoir 3/ ; les artefacts P2 sont dans Devoir 2/
NOTEBOOK_DIR  = Path().resolve()          # répertoire du notebook
P2_ROOT       = NOTEBOOK_DIR.parent / "Devoir 2"   # racine projet P2
VARIANT       = "temporal_pre_split"       # sous-ensemble D choisi
DATA_DIR      = P2_ROOT / "data" / "joining" / VARIANT

print(f"Notebook dir : {NOTEBOOK_DIR}")
print(f"P2 root      : {P2_ROOT}")
print(f"Data dir     : {DATA_DIR}")
assert DATA_DIR.exists(), f"Répertoire introuvable : {DATA_DIR}"
print("✓ Répertoire de données trouvé.")

In [ ]:
TOP_N = 10
COLD_START_THRESHOLD = 3
RELEVANCE_THRESHOLD  = 0.0   # tout item du test est pertinent

print(f"TOP_N={TOP_N}  COLD_START_THRESHOLD={COLD_START_THRESHOLD}")

---
## 1. Fonctions d'évaluation communes (réutilisées de P2)

Ces fonctions sont copiées/adaptées depuis `scripts/task4/evaluation.py` du projet P2 pour rendre ce notebook autonome.

In [ ]:
# ── Ground truth ────────────────────────────────────────────────────────────

def load_ground_truth(
    data_dir: Path,
) -> Tuple[Dict[str, set], Dict[str, Dict[str, float]]]:
    """Construit le ground truth depuis le jeu de test.
    Retourne gt_binary = {user_id: set(asin)} et
              gt_graded = {user_id: {asin: rating}}.
    """
    test_df = pd.read_parquet(
        data_dir / "test_interactions.parquet",
        columns=["user_id", "parent_asin", "rating"],
    )
    gt_binary: Dict[str, set] = {}
    gt_graded: Dict[str, Dict[str, float]] = {}
    for uid, g in test_df.groupby("user_id"):
        items   = set(g["parent_asin"])
        ratings = dict(zip(g["parent_asin"], g["rating"]))
        if RELEVANCE_THRESHOLD > 0:
            items = {a for a, r in ratings.items() if r >= RELEVANCE_THRESHOLD}
        if items:
            gt_binary[uid] = items
            gt_graded[uid] = ratings
    return gt_binary, gt_graded


def load_train_activity(data_dir: Path) -> pd.DataFrame:
    """Nombre d'interactions par utilisateur dans le train."""
    train_df = pd.read_parquet(
        data_dir / "train_interactions.parquet",
        columns=["user_id", "parent_asin"],
    )
    return (
        train_df.groupby("user_id")["parent_asin"]
        .count().rename("nb_train").reset_index()
    )


# ── Segmentation utilisateurs ───────────────────────────────────────────────

def build_user_segments(
    activity: pd.DataFrame,
) -> Tuple[Dict[str, set], Dict]:
    q1 = int(activity["nb_train"].quantile(0.25))
    q3 = int(activity["nb_train"].quantile(0.75))
    segments = {
        "peu_actifs":        set(activity[activity["nb_train"] < q1]["user_id"]),
        "moderement_actifs": set(activity[(activity["nb_train"] >= q1) & (activity["nb_train"] <= q3)]["user_id"]),
        "tres_actifs":       set(activity[activity["nb_train"] > q3]["user_id"]),
        "cold_start":        set(activity[activity["nb_train"] <= COLD_START_THRESHOLD]["user_id"]),
    }
    info = {
        "peu_actifs":        f"< Q1 ({q1})",
        "moderement_actifs": f"Q1–Q3 ({q1}–{q3})",
        "tres_actifs":       f"> Q3 ({q3})",
        "cold_start":        f"≤ {COLD_START_THRESHOLD}",
        "q1": q1, "q3": q3,
    }
    return segments, info


# ── Métriques individuelles ─────────────────────────────────────────────────

def precision_at_n(rec: List[str], relevant: set, n: int) -> float:
    return sum(1 for a in rec[:n] if a in relevant) / n

def recall_at_n(rec: List[str], relevant: set, n: int) -> float:
    if not relevant: return 0.0
    return sum(1 for a in rec[:n] if a in relevant) / len(relevant)

def ndcg_at_n(rec: List[str], graded: Dict[str, float], n: int) -> float:
    dcg = sum((graded.get(a, 0.0) / 5.0) / np.log2(i + 2) for i, a in enumerate(rec[:n]))
    ideal = sorted([r / 5.0 for r in graded.values()], reverse=True)[:n]
    idcg  = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def ndcg_binary_at_n(rec: List[str], relevant: set, n: int) -> float:
    dcg  = sum((1.0 if a in relevant else 0.0) / np.log2(i + 2) for i, a in enumerate(rec[:n]))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), n)))
    return dcg / idcg if idcg > 0 else 0.0

def ap_at_n(rec: List[str], relevant: set, n: int) -> float:
    hits, total = 0, 0.0
    for i, a in enumerate(rec[:n]):
        if a in relevant:
            hits += 1; total += hits / (i + 1)
    return total / min(len(relevant), n) if relevant else 0.0

def mrr_at_n(rec: List[str], relevant: set, n: int) -> float:
    for i, a in enumerate(rec[:n]):
        if a in relevant: return 1.0 / (i + 1)
    return 0.0

def user_metrics(
    rec: List[str], gt_bin: set, gt_grad: Dict[str, float], n: int
) -> Dict[str, float]:
    return {
        "precision":   precision_at_n(rec, gt_bin, n),
        "recall":      recall_at_n(rec, gt_bin, n),
        "ndcg":        ndcg_at_n(rec, gt_grad, n),
        "ndcg_binary": ndcg_binary_at_n(rec, gt_bin, n),
        "ap":          ap_at_n(rec, gt_bin, n),
        "mrr":         mrr_at_n(rec, gt_bin, n),
        "hit":         1.0 if any(a in gt_bin for a in rec[:n]) else 0.0,
    }


# ── Évaluation globale ──────────────────────────────────────────────────────

def evaluate_top_n(
    top_n_indices: np.ndarray,
    user_ids: np.ndarray,
    item_ids: np.ndarray,
    gt_binary: Dict[str, set],
    gt_graded: Dict[str, Dict[str, float]],
    n: int = TOP_N,
    label: str = "",
) -> Tuple[Dict[str, float], pd.DataFrame]:
    """Évalue une liste top-N et retourne métriques agrégées + DataFrame par user."""
    records = []
    for u_row, uid in enumerate(user_ids):
        if uid not in gt_binary:
            continue
        indices = top_n_indices[u_row]
        valid   = indices[indices >= 0]
        rec     = [item_ids[i] for i in valid]
        m = user_metrics(rec, gt_binary[uid], gt_graded.get(uid, {}), n)
        m["user_id"] = uid
        records.append(m)

    df = pd.DataFrame(records)
    agg = {
        "n_users":       len(df),
        "Precision@N":   round(df["precision"].mean(), 6),
        "Recall@N":      round(df["recall"].mean(), 6),
        "NDCG@N":        round(df["ndcg"].mean(), 6),
        "NDCG_bin@N":    round(df["ndcg_binary"].mean(), 6),
        "MAP@N":         round(df["ap"].mean(), 6),
        "MRR@N":         round(df["mrr"].mean(), 6),
        "HitRate@N":     round(df["hit"].mean(), 6),
    }
    if label:
        print(f"  [{label}] "
              f"P@{n}={agg['Precision@N']:.4f}  R@{n}={agg['Recall@N']:.4f}  "
              f"NDCG@{n}={agg['NDCG@N']:.4f}  HR@{n}={agg['HitRate@N']:.4f}  "
              f"({agg['n_users']} users)")
    return agg, df


def evaluate_by_segment(
    user_df: pd.DataFrame,
    segments: Dict[str, set],
    n: int = TOP_N,
) -> Dict[str, Dict]:
    results = {}
    for seg, users in segments.items():
        sub = user_df[user_df["user_id"].isin(users)]
        if len(sub) == 0:
            results[seg] = {k: 0.0 for k in ["n_users","Precision@N","Recall@N",
                                               "NDCG@N","NDCG_bin@N","MAP@N","MRR@N","HitRate@N"]}
            results[seg]["n_users"] = 0
            continue
        results[seg] = {
            "n_users":       len(sub),
            "Precision@N":   round(sub["precision"].mean(), 6),
            "Recall@N":      round(sub["recall"].mean(), 6),
            "NDCG@N":        round(sub["ndcg"].mean(), 6),
            "NDCG_bin@N":    round(sub["ndcg_binary"].mean(), 6),
            "MAP@N":         round(sub["ap"].mean(), 6),
            "MRR@N":         round(sub["mrr"].mean(), 6),
            "HitRate@N":     round(sub["hit"].mean(), 6),
        }
    return results


# ── ILD et couverture catalogue ─────────────────────────────────────────────

def compute_ild(
    top_n_indices: np.ndarray,
    user_ids: np.ndarray,
    item_matrix: csr_matrix,
    gt_users: set,
) -> Tuple[float, pd.DataFrame]:
    """ILD = 1 − moyenne des similarités cosinus intra-liste (vecteurs TF-IDF)."""
    t0 = time.perf_counter()
    normed = normalize(item_matrix, norm="l2", copy=True)
    records = []
    for u_row, uid in enumerate(user_ids):
        if uid not in gt_users:
            continue
        valid = top_n_indices[u_row]
        valid = valid[(valid >= 0) & (valid < item_matrix.shape[0])]
        if len(valid) < 2:
            records.append({"user_id": uid, "ild": 1.0})
            continue
        vecs = normed[valid]
        sim  = sk_cosine(vecs)
        idx  = np.triu_indices(sim.shape[0], k=1)
        records.append({"user_id": uid, "ild": round(1.0 - float(sim[idx].mean()), 6)})
    df = pd.DataFrame(records)
    mean_ild = float(df["ild"].mean()) if len(df) > 0 else 0.0
    print(f"    ILD={mean_ild:.4f} ({len(df)} users, {time.perf_counter()-t0:.1f}s)")
    return round(mean_ild, 6), df


def catalog_coverage(top_n_indices: np.ndarray, n_items: int) -> float:
    valid   = top_n_indices[top_n_indices >= 0]
    n_uniq  = len(np.unique(valid))
    return round(n_uniq / n_items, 6) if n_items > 0 else 0.0


print("Fonctions d'évaluation chargées.")

---
## 2. Chargement des artefacts communs

In [ ]:
t0 = time.perf_counter()

# --- ID maps ---
user_ids  = np.load(DATA_DIR / "user_ids.npy",   allow_pickle=True)
item_ids  = np.load(DATA_DIR / "item_ids.npy",   allow_pickle=True)
item_titles = np.load(DATA_DIR / "item_titles.npy", allow_pickle=True)
print(f"user_ids  : {len(user_ids):,}")
print(f"item_ids  : {len(item_ids):,}")

# --- Ground truth (test set) ---
gt_binary, gt_graded = load_ground_truth(DATA_DIR)
print(f"Users dans le test : {len(gt_binary):,}")

# --- Activité train + segmentation ---
activity = load_train_activity(DATA_DIR)
segments, seg_info = build_user_segments(activity)
print(f"Segmentation (Q1={seg_info['q1']}, Q3={seg_info['q3']}) :")
for seg, desc in seg_info.items():
    if seg in ("q1", "q3"): continue
    n_test = len(segments[seg] & set(gt_binary.keys()))
    print(f"  {seg:<22}: {len(segments[seg]):,} users ({n_test:,} dans le test)")

# --- Matrice TF-IDF items (pour ILD) ---
item_matrix = load_npz(DATA_DIR / "books_representation_sparse.npz")
print(f"\nMatrice TF-IDF items : {item_matrix.shape[0]:,} × {item_matrix.shape[1]:,}")
print(f"Chargement total : {time.perf_counter()-t0:.1f}s")

---
## 3. Tâche 0 – Évaluation du système content-based (P2)

Le recommandeur basé sur le contenu du projet P2 (**scripts/task1/similarity.py**) a déjà produit les fichiers `top_n_indices_10.npy` dans `data/joining/temporal_pre_split/`. On charge ces indices et on évalue les métriques top-N.

In [ ]:
print("=" * 65)
print("  Tâche 0 — Évaluation filtrage basé sur le contenu (P2)")
print("=" * 65)
t0_start = time.perf_counter()

# Charger les recommandations content-based produites par P2
cb_topn_path = DATA_DIR / f"top_n_indices_{TOP_N}.npy"
assert cb_topn_path.exists(), f"Fichier P2 introuvable : {cb_topn_path}"
cb_top_n = np.load(cb_topn_path)
print(f"top_n_indices shape : {cb_top_n.shape}  (users × top-N)")

# Métriques globales
print("\n[Métriques globales]")
cb_agg, cb_user_df = evaluate_top_n(
    cb_top_n, user_ids, item_ids, gt_binary, gt_graded,
    n=TOP_N, label="Content-Based"
)

# ILD
print("\n[ILD – diversité intra-liste]")
cb_ild, cb_ild_df = compute_ild(cb_top_n, user_ids, item_matrix, set(gt_binary.keys()))

# Couverture catalogue
cb_cov = catalog_coverage(cb_top_n, len(item_ids))
print(f"    Couverture catalogue : {cb_cov:.2%}")

# Métriques par segment
print("\n[Métriques par segment]")
cb_seg = evaluate_by_segment(cb_user_df, segments, TOP_N)
for seg, m in cb_seg.items():
    if m["n_users"] == 0: continue
    print(f"  {seg:<22}: N={m['n_users']:>5}  "
          f"NDCG={m['NDCG@N']:.4f}  HR={m['HitRate@N']:.4f}  R={m['Recall@N']:.4f}")

print(f"\nTâche 0 terminée en {time.perf_counter()-t0_start:.1f}s")

In [ ]:
# Tableau récapitulatif Tâche 0
cb_summary = pd.DataFrame([{
    "Approche":          "Content-Based (TF-IDF)",
    f"P@{TOP_N}":        cb_agg["Precision@N"],
    f"R@{TOP_N}":        cb_agg["Recall@N"],
    f"NDCG@{TOP_N}":     cb_agg["NDCG@N"],
    f"NDCG_bin@{TOP_N}": cb_agg["NDCG_bin@N"],
    f"MAP@{TOP_N}":      cb_agg["MAP@N"],
    f"MRR@{TOP_N}":      cb_agg["MRR@N"],
    f"HR@{TOP_N}":       cb_agg["HitRate@N"],
    "ILD":               cb_ild,
    "Couverture":        f"{cb_cov:.2%}",
    "N users": cb_agg["n_users"],
}])
print(f"\n{'─'*65}")
print(f"  Résumé Tâche 0 — Filtrage contenu (top-{TOP_N})")
print(f"{'─'*65}")
print(cb_summary.to_string(index=False))

---
## 4. Tâche 1 – Filtrage collaboratif item-based

### Choix de conception

On implémente un **filtrage collaboratif basé sur les items** (*item-based CF*) avec similarité cosinus sur la matrice de notes utilisateur–item.

**Justification** :
- Les données Amazon Books sont très éparses (peu de ratings par utilisateur) : l'item-based CF est plus stable que l'user-based CF car les profils items (agrégés sur de nombreux utilisateurs) sont plus robustes que les profils utilisateurs (peu d'interactions).
- Même infrastructure scipy.sparse que dans P2 → aucune dépendance nouvelle.
- Scalable : la matrice de similarité items n'est calculée qu'une fois (offline).

**Algorithme** :
1. Construire la matrice R (users × items) avec les ratings du train.
2. Calculer la similarité cosinus entre items (sur les colonnes de R, i.e. sur les profils de co-notation).
3. Pour chaque utilisateur, scorer les items non vus : score(u, i) = Σ_{j ∈ historique(u)} R[u,j] × sim(i,j).
4. Retourner le Top-N après masquage des items déjà notés.

### 4.1 Construction de la matrice utilisateur–item

In [ ]:
print("=" * 65)
print("  Tâche 1 — Construction de la matrice utilisateur–item")
print("=" * 65)
t1_start = time.perf_counter()

train_df = pd.read_parquet(
    DATA_DIR / "train_interactions.parquet",
    columns=["user_id", "parent_asin", "rating"],
)
print(f"Train interactions : {len(train_df):,} lignes")

# Index compacts user → row / item → col
user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {a: j for j, a in enumerate(item_ids)}

rows = train_df["user_id"].map(user_to_idx)
cols = train_df["parent_asin"].map(item_to_idx)
vals = train_df["rating"].astype(np.float32)

valid_mask = rows.notna() & cols.notna()
rows = rows[valid_mask].astype(int)
cols = cols[valid_mask].astype(int)
vals = vals[valid_mask]
print(f"Paires valides : {valid_mask.sum():,}")

R = csr_matrix(
    (vals.values, (rows.values, cols.values)),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.float32,
)
print(f"Matrice R (users × items) : {R.shape}  "
      f"nnz={R.nnz:,}  densité={R.nnz / (R.shape[0]*R.shape[1]):.5f}")
print(f"Construit en {time.perf_counter()-t1_start:.1f}s")

del train_df, rows, cols, vals
gc.collect()

### 4.2 Calcul de la similarité item–item (cosinus, par batchs)

In [ ]:
def compute_item_similarity_batched(
    R: csr_matrix,
    batch_size: int = 500,
) -> np.ndarray:
    """
    Calcule la similarité cosinus entre items à partir de la matrice R.
    R est de forme (users × items) ; on travaille sur R.T (items × users).
    Retourne une matrice dense (n_items × n_items) en float32.
    
    Pour limiter la mémoire on calcule par batchs de lignes et on stocke
    directement la matrice de similarité complète.
    """
    Rt = R.T  # items × users
    n_items = Rt.shape[0]
    # Normaliser une seule fois
    Rt_normed = normalize(Rt, norm="l2", copy=True)
    
    item_sim = np.empty((n_items, n_items), dtype=np.float32)
    t0 = time.perf_counter()
    for start in range(0, n_items, batch_size):
        end = min(start + batch_size, n_items)
        block = sk_cosine(Rt_normed[start:end], Rt_normed)
        item_sim[start:end] = block.astype(np.float32, copy=False)
        if (start // batch_size) % 5 == 0:
            pct = end / n_items * 100
            print(f"    {end:>6}/{n_items} ({pct:5.1f}%)  {time.perf_counter()-t0:.0f}s")
    
    np.fill_diagonal(item_sim, 0.0)  # exclure l'auto-similarité
    return item_sim


print("Calcul de la similarité item–item (cosinus)...")
t_sim = time.perf_counter()
item_sim = compute_item_similarity_batched(R, batch_size=500)
print(f"Similarité item–item calculée en {time.perf_counter()-t_sim:.1f}s")
print(f"item_sim shape : {item_sim.shape}  dtype={item_sim.dtype}")
gc.collect()

### 4.3 Génération des Top-N recommandations

In [ ]:
def generate_cf_top_n(
    R: csr_matrix,
    item_sim: np.ndarray,
    top_n: int = TOP_N,
    batch_size: int = 256,
) -> np.ndarray:
    """
    Pour chaque utilisateur, calcule le score item-based CF et retourne le Top-N.

    score(u, i) = Σ_{j ∈ notes(u)} R[u,j] × sim(i,j)

    Implémentation matricielle :
        scores = R × item_sim  →  (users × items)
    Puis on masque les items déjà notés et on prend le top-N.
    """
    n_users, n_items = R.shape
    top_idx = np.empty((n_users, top_n), dtype=np.int32)
    t0 = time.perf_counter()

    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)
        # Score = sous-matrice dense (batch_users × n_items)
        R_block = R[start:end]          # sparse
        scores  = R_block.dot(item_sim).astype(np.float32)  # dense

        # Masquer items déjà notés dans le train
        r_block_coo = R_block.tocoo()
        local_rows  = r_block_coo.row   # lignes locales (0..batch_size-1)
        seen_cols   = r_block_coo.col
        scores[local_rows, seen_cols] = -np.inf

        # Top-N par argpartition (plus rapide que argsort complet)
        kth  = top_n - 1
        part = np.argpartition(-scores, kth=kth, axis=1)[:, :top_n]
        rr   = np.arange(scores.shape[0])[:, None]
        ord_ = np.argsort(-scores[rr, part], axis=1)
        top_idx[start:end] = part[rr, ord_]

        if (start // batch_size) % 10 == 0:
            pct = end / n_users * 100
            print(f"    {end:>6}/{n_users} ({pct:5.1f}%)  {time.perf_counter()-t0:.0f}s")

        del scores, R_block

    return top_idx


print("Génération des recommandations item-based CF...")
t_gen = time.perf_counter()
cf_top_n = generate_cf_top_n(R, item_sim, top_n=TOP_N, batch_size=256)
print(f"Top-{TOP_N} généré en {time.perf_counter()-t_gen:.1f}s")
print(f"cf_top_n shape : {cf_top_n.shape}")

# Sauvegarde des indices CF pour la Tâche 2 (graphe de connaissances)
cf_out = DATA_DIR / f"cf_itembased_top_n_{TOP_N}.npy"
np.save(cf_out, cf_top_n)
print(f"Indices CF sauvegardés → {cf_out}")

gc.collect()

### 4.4 Évaluation du filtrage collaboratif

In [ ]:
print("\n[Métriques globales — CF item-based]")
cf_agg, cf_user_df = evaluate_top_n(
    cf_top_n, user_ids, item_ids, gt_binary, gt_graded,
    n=TOP_N, label="CF item-based"
)

print("\n[ILD – diversité intra-liste]")
cf_ild, cf_ild_df = compute_ild(cf_top_n, user_ids, item_matrix, set(gt_binary.keys()))

cf_cov = catalog_coverage(cf_top_n, len(item_ids))
print(f"    Couverture catalogue : {cf_cov:.2%}")

print("\n[Métriques par segment]")
cf_seg = evaluate_by_segment(cf_user_df, segments, TOP_N)
for seg, m in cf_seg.items():
    if m["n_users"] == 0: continue
    print(f"  {seg:<22}: N={m['n_users']:>5}  "
          f"NDCG={m['NDCG@N']:.4f}  HR={m['HitRate@N']:.4f}  R={m['Recall@N']:.4f}")

print(f"\nTâche 1 terminée en {time.perf_counter()-t1_start:.1f}s")

---
## 5. Tableau comparatif – Content-Based vs Collaboratif

In [ ]:
import pandas as pd

def make_row(name, agg, ild, cov):
    return {
        "Approche":          name,
        f"P@{TOP_N}":        agg["Precision@N"],
        f"R@{TOP_N}":        agg["Recall@N"],
        f"NDCG@{TOP_N}":     agg["NDCG@N"],
        f"NDCG_bin@{TOP_N}": agg["NDCG_bin@N"],
        f"MAP@{TOP_N}":      agg["MAP@N"],
        f"MRR@{TOP_N}":      agg["MRR@N"],
        f"HR@{TOP_N}":       agg["HitRate@N"],
        "ILD":               ild,
        "Couverture":        f"{cov:.2%}",
        "N users":           agg["n_users"],
    }

comparison = pd.DataFrame([
    make_row("Content-Based (TF-IDF)", cb_agg, cb_ild, cb_cov),
    make_row("CF item-based (cosinus)", cf_agg, cf_ild, cf_cov),
])

print("="*65)
print(f"  COMPARAISON GLOBALE — Tâche 0 vs Tâche 1 (Top-{TOP_N})")
print(f"  Sous-ensemble D : {VARIANT}")
print("="*65)
print(comparison.to_string(index=False))

In [ ]:
# Tableau croisé par segment
seg_order  = ["peu_actifs", "moderement_actifs", "tres_actifs", "cold_start"]
seg_labels = {
    "peu_actifs":        "Peu actifs",
    "moderement_actifs": "Modérément actifs",
    "tres_actifs":       "Très actifs",
    "cold_start":        "Cold-start",
}

rows_seg = []
for seg in seg_order:
    cb_m = cb_seg.get(seg, {})
    cf_m = cf_seg.get(seg, {})
    rows_seg.append({
        "Segment":             seg_labels[seg],
        "N users":             cb_m.get("n_users", 0),
        f"CB NDCG@{TOP_N}":    cb_m.get("NDCG@N", 0),
        f"CF NDCG@{TOP_N}":    cf_m.get("NDCG@N", 0),
        f"CB HR@{TOP_N}":      cb_m.get("HitRate@N", 0),
        f"CF HR@{TOP_N}":      cf_m.get("HitRate@N", 0),
        f"CB Recall@{TOP_N}":  cb_m.get("Recall@N", 0),
        f"CF Recall@{TOP_N}":  cf_m.get("Recall@N", 0),
    })

seg_df = pd.DataFrame(rows_seg)
print("\nComparaison par segment d'utilisateurs :")
print(seg_df.to_string(index=False))

In [ ]:
# Visualisation barres groupées
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 10})

metrics_plot = [f"P@{TOP_N}", f"R@{TOP_N}", f"NDCG@{TOP_N}",
                f"MAP@{TOP_N}", f"MRR@{TOP_N}", f"HR@{TOP_N}", "ILD"]
cb_vals = [cb_agg["Precision@N"], cb_agg["Recall@N"], cb_agg["NDCG@N"],
           cb_agg["MAP@N"], cb_agg["MRR@N"], cb_agg["HitRate@N"], cb_ild]
cf_vals = [cf_agg["Precision@N"], cf_agg["Recall@N"], cf_agg["NDCG@N"],
           cf_agg["MAP@N"], cf_agg["MRR@N"], cf_agg["HitRate@N"], cf_ild]

x = np.arange(len(metrics_plot))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, cb_vals, w, label="Content-Based (TF-IDF)", color="steelblue")
ax.bar(x + w/2, cf_vals, w, label="CF item-based (cosinus)", color="coral")

ax.set_xticks(x)
ax.set_xticklabels(metrics_plot)
ax.set_ylabel("Score")
ax.set_title(f"Comparaison Content-Based vs Collaboratif — Top-{TOP_N} ({VARIANT})")
ax.legend()
ax.set_ylim(0, max(max(cb_vals), max(cf_vals)) * 1.25)

for bar in ax.patches:
    h = bar.get_height()
    if h > 0:
        ax.annotate(f"{h:.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / "comparison_tasks01.png", dpi=120)
plt.show()
print("Graphique sauvegardé → comparison_tasks01.png")

---
## 6. Discussion

### 6.1 Interprétation des métriques

| Dimension | Content-Based | CF item-based | Avantage |
|-----------|:-------------:|:-------------:|:---------:|
| Pertinence (NDCG@10) | cf. résultats | cf. résultats | ─ |
| Couverture (Recall@10) | cf. résultats | cf. résultats | ─ |
| Capacité à trouver au moins 1 item (HR@10) | cf. résultats | cf. résultats | ─ |
| Diversité (ILD) | cf. résultats | cf. résultats | ─ |
| Couverture catalogue | cf. résultats | cf. résultats | ─ |

*(Les cellules marquées ─ seront complétées avec les valeurs numériques une fois le notebook exécuté.)*

### 6.2 Analyse comparative

**Filtrage basé sur le contenu (P2)** :
- Recommande des items dont le contenu textuel (TF-IDF titre, description, catégories) ressemble au profil agrégé de l'utilisateur.
- Avantages : pas de problème de cold-start item (tout item avec métadonnées peut être recommandé) ; recommandations explicables.
- Limites : surspécialisation (bulle de filtre) ; ignore les signaux de popularité collective ; insensible à la qualité perçue par d'autres utilisateurs.

**Filtrage collaboratif item-based** :
- Recommande des items fréquemment appréciés ensemble par la communauté d'utilisateurs similaires.
- Avantages : exploite la sagesse collective ; peut découvrir des items thématiquement différents mais comportementalement liés ; moins de surspécialisation.
- Limites : cold-start item (les nouveaux items sans interactions ne peuvent pas être recommandés) ; cold-start utilisateur (peu d'interactions → profil faible) ; scalabilité de la matrice item–item (O(n_items²)).

### 6.3 Segments d'utilisateurs

- **Très actifs** : les deux approches bénéficient d'un historique riche. Le CF item-based peut en tirer plus d'avantages car la somme pondérée des similarités est plus robuste.
- **Cold-start (≤ 3 interactions)** : le CB souffre d'un profil peu représentatif ; le CF souffre d'un historique trop court pour trouver des items similaires bien notés. Les deux approches sont donc pénalisées sur ce segment.
- **Éclectiques** : le CB tendra à recommander une liste hétérogène (profil moyen diluant les intérêts) ; le CF exploitera les co-occurrences spécifiques.

### 6.4 Vers la Tâche 2

La Tâche 2 (graphe de connaissances) vise à enrichir les recommandations par inférence sémantique. On attend :
- Une meilleure gestion du cold-start (raisonnement sur les catégories / auteurs).
- Des recommandations plus explicables (chemin d'inférence dans le graphe).
- Potentiellement une diversité accrue grâce aux liens sémantiques entre entités.

In [ ]:
# Export JSON du tableau comparatif pour réutilisation dans la Tâche 2
import json

export = {
    "variant": VARIANT,
    "top_n": TOP_N,
    "content_based": {
        **cb_agg,
        "ild": cb_ild,
        "catalog_coverage": cb_cov,
        "by_segment": cb_seg,
    },
    "cf_item_based": {
        **cf_agg,
        "ild": cf_ild,
        "catalog_coverage": cf_cov,
        "by_segment": cf_seg,
    },
}

def _to_json_safe(obj):
    if isinstance(obj, dict):   return {k: _to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [_to_json_safe(v) for v in obj]
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

out_path = NOTEBOOK_DIR / "evaluation_tasks01.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(_to_json_safe(export), f, indent=2, ensure_ascii=False)
print(f"Résultats exportés → {out_path}")